## Preprocesamiento esencial de datos

In [1]:
import torch
import pandas as pd
import json
from modelado.dataset_form import train_transform, val_transform, SimpleDataset

## Carga de datos

In [2]:
tile_meta = pd.read_csv('data/tile_meta.csv')
with open('data/polygons.jsonl', 'r') as f:
    annotations = [json.loads(line) for line in f]
annotations_dict = {ann['id']: ann for ann in annotations}

## Distribuir en train, test, val

In [3]:
from sklearn.model_selection import train_test_split
all_tiles = tile_meta['id'].tolist()
train_tiles, temp = train_test_split(all_tiles, test_size=0.3, random_state=42)
val_tiles, test_tiles = train_test_split(temp, test_size=0.5, random_state=42)

## Guardar data para modelado

In [4]:
train_dataset = SimpleDataset(train_tiles, 'train/', annotations_dict, train_transform)
val_dataset = SimpleDataset(val_tiles, 'train/', annotations_dict, val_transform)
test_dataset = SimpleDataset(test_tiles, 'train/', annotations_dict, val_transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=8, shuffle=False)

print(f"Listo: {len(train_tiles)} train, {len(val_tiles)} val, {len(test_tiles)} test")

Listo: 4923 train, 1055 val, 1055 test


In [5]:
import pickle
pickle.dump({'train': train_tiles, 'val': val_tiles, 'test': test_tiles, 
             'annotations': annotations_dict}, open('modelado/data_minimal.pkl', 'wb'))